# Trabalho Prático de Otimização - Otimização Não-Linear Irrestrita

### Imports

In [2]:
from typing import List
import numpy as np

## Funções

In [18]:
def func_a(x:np.ndarray)->float|None:

    if len(x) != 7:
        return None

    y = 0

    for i in range(0,6):
        y += pow(100*(x[i+1] - pow(x[i],2)),2) + pow(1-x[i],2)

    return y

In [44]:
def func_b(x:np.ndarray)->float|None:

    if len(x) != 100:
        return None

    y = 0

    for i in range(0, 99):
        y += pow(x[i], 4) - 16*pow(x[i], 2) + 5*x[i]

    return y

In [21]:
def func_c(x:np.ndarray)->float|None:

    if len(x) != 2:
        return None

    y = pow(pow(x[0], 2) + x[1] - 11, 2) + pow(x[0] + pow(x[1], 2) - 7,2)

    return y

### Gradientes

In [107]:
def gradient_func_a(x: np.ndarray) -> np.ndarray | None:
    if len(x) != 7:
        return None

    grad = np.zeros_like(x)

    grad[0] = -400 * x[0] * (-x[0]**2 + x[1]) + 2 * x[0] - 2

    for i in range(1, 5):
        grad[i] = -200 * x[i-1]**2 - 400 * x[i] * (-x[i]**2 + x[i+1]) + 202 * x[i] - 2

    grad[5] = -200 * x[5] * (-x[5]**2) + 200 * x[6]

    return grad


In [43]:
def gradient_func_b(x:np.ndarray)->np.ndarray[float]|None:

    if len(x) != 100:
        return None

    grad = np.zeros_like(x)

    for i in range(100):
        grad[i] = 4 * x[i]**3 - 32 * x[i] + 5

    return grad

In [20]:
def gradient_func_c(x:np.ndarray)->np.ndarray[float]|None:

    if len(x) != 2:
            return None

    grad = np.zeros_like(x)

    grad[0] = 4 * (pow(x[0], 2) + x[1] - 11) * x[0] + 2 * (x[0] + pow(x[1], 2) - 7)
    grad[1] = 2 * (pow(x[0], 2) + x[1] - 11) + 4 * (x[0] + pow(x[1], 2) - 7)* x[1]

    return grad


### Hessianas

In [106]:
def hessian_func_a(x: np.ndarray) -> np.ndarray | None:
    if len(x) != 7:
        return None

    hess = np.zeros((6, 6))

    hess[0, 0] = -400 * (-3 * x[0]**2 + x[1]) + 2
    hess[0, 1] = -400 * x[0]

    for i in range(1, 5):
        hess[i, i] = -400 * (-3 * x[i]**2 + x[i+1]) + 202
        hess[i, i-1] = hess[i-1, i] = -400 * x[i]

    hess[5, 5] = -200 * (-3 * x[5]**2) + 200

    return hess

In [13]:
def hessian_func_b(x:np.ndarray)->np.ndarray|None:
    if len(x) != 100:
            return None

    hess = np.zeros((len(x), len(x)))

    for i in range(len(x)):
        hess[i, i] = 12 * x[i]**2 - 32

    return hess

In [4]:
def hessian_func_c(x:np.ndarray)->np.ndarray|None:
    if len(x) != 2:
        return None

    hess = np.zeros((2, 2))

    hess[0, 0] = 2 * (3 * x[0]**2 + 2 * x[1] - 11)
    hess[0, 1] = 2 * (2 * x[0] + 2 * x[1])

    hess[1, 0] = 2 * (2 * x[0] + 2 * x[1])
    hess[1, 1] = 2 * (2 * x[0] + 3 * x[1]**2 - 7)

    return hess

In [109]:
enum_functions = {'a':[func_a, gradient_func_a, hessian_func_a], 'b':[func_b, gradient_func_b, hessian_func_b], 'c':[func_c, gradient_func_c, hessian_func_c]}

## Cálculo do Passo

### Armijo

In [75]:
def armijo_rule(function_id: str, x: np.ndarray, d: np.ndarray, mi: float, gama: float = 0.8):
    t = 1
    term = np.dot(enum_functions[function_id][1](x).T, d)

    while enum_functions[function_id][0](x + t * d) > enum_functions[function_id][0](x) + mi * t * term:
        t = gama * t

    return t


### Cálculos de d

In [48]:
def d_gradient(function_id:str,x:np.ndarray):
    return -enum_functions[function_id][1](x)

In [49]:
def d_newton(function_id:str,x:np.ndarray):
    h = enum_functions[function_id][2](x)
    grad = enum_functions[function_id][1](x)
    return - np.dot(np.linalg.inv(h), grad)

In [50]:
enum_directions = {'gradient': d_gradient, 'newton': d_newton}

### Cálculos de H

In [192]:
def h_dfp(function_id:str,x:np.ndarray,new_x:np.ndarray,h:np.ndarray,tol:float):
    p = new_x - x
    q = enum_functions[function_id][1](new_x) - enum_functions[function_id][1](x)

    term1 = (p * p.T) / (p.T @ q)
    term2 = (h @ q @ q.T @ h) / (q.T @ h @ q)

    return term1 - term2


In [193]:
def h_bfgs(function_id:str,x:np.ndarray,new_x:np.ndarray,h:np.ndarray,tol:float):
    p = new_x - x
    q = enum_functions[function_id][1](new_x) - enum_functions[function_id][1](x)

    pq = np.dot(p.T, q)
    if np.all(abs(pq) < tol):
        return h*0
    qHq = np.dot(np.dot(q.T, h), q)

    term1 = (1 + qHq / pq) * (np.dot(p, p.T) / pq)
    term2 = (np.dot(p, np.dot(q.T, h)) + np.dot(h, q) @ p.T) / pq

    return term1 - term2

In [194]:
enum_hessiana = {'dfp': h_dfp, 'bfgs': h_bfgs}

#### Algoritmo de Descida (para gradiente e newton)

In [168]:
def base_descent(function_id:str,x:np.ndarray,direction_id:str,tol:float):

    k = 0

    while np.all(abs(enum_functions[function_id][1](x)) > tol):
        dk = enum_directions[direction_id](function_id, x)
        tk = armijo_rule(function_id, x, dk, mi=.25)
        if np.all(abs(tk*dk) < tol):
            return x
        x += tk*dk
        k += 1
        print(k, x, enum_functions[function_id][0](x))
    return x

#### Algoritmo de Descida (para quase-newton)

In [219]:
def quasi_newton_descent(function_id:str,hessiana_id:str,x:np.ndarray,tol:float):

    k = 0
    h = enum_functions[function_id][2](x)
    while np.all(abs(enum_functions[function_id][1](x)) > tol):
        dk = -h @ enum_functions[function_id][1](x)
        tk = armijo_rule(function_id, x, dk, mi=.25)
        h += enum_hessiana[hessiana_id](function_id, x, x+tk*dk, h, tol)
        if np.all(abs(tk*dk) < tol):
            return x
        x += tk*dk
        k += 1
    return x

In [221]:
start_point = np.zeros(2)
print(quasi_newton_descent('c', 'bfgs', start_point, 0.00001 ))


[-3.19205948 -3.69347769]
